# YouTube Relaxing Music Video Generator
Generate 10-minute relaxing videos with background footage

In [ ]:
# Install dependencies
!pip install requests python-dotenv -q

In [ ]:
import os, requests, subprocess
from pathlib import Path

OUTPUT = Path('/content/output')
OUTPUT.mkdir(parents=True, exist_ok=True)

# Download stock footage from Pexels
def download_stock(query='relaxing nature', count=3):
    api_key = os.environ.get('PEXELS_API_KEY', 'demo')
    if api_key == 'demo':
        print('Using demo mode - no API key configured')
        return []
    
    headers = {'Authorization': api_key}
    resp = requests.get(
        'https://api.pexels.com/v1/search',
        headers=headers,
        params={'query': query, 'per_page': count, 'orientation': 'landscape'}
    )
    
    videos = []
    if resp.status_code == 200:
        for v in resp.json().get('videos', [])[:count]:
            url = v['video_files'][0]['link']
            out = OUTPUT / f'stock_{len(videos)}.mp4'
            video_data = requests.get(url).content
            out.write_bytes(video_data)
            videos.append(str(out))
    return videos

In [ ]:
# Generate relaxing video
def create_video():
    print('Creating 10-minute relaxing video...')
    
    # Download stock footage
    stock = download_stock('relaxing nature', 3)
    
    output = OUTPUT / 'relaxing_music_10min.mp4'
    
    if stock:
        # Create video with stock footage
        list_file = OUTPUT / 'list.txt'
        list_file.write_text('\n'.join(f"file '{s}'" for s in stock))
        
        cmd = ['ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', str(list_file),
               '-vf', f"drawtext=text='Relaxing Music':fontsize=48:fontcolor=white:x=(w-text_w)/2:y=h-100",
               '-c:v', 'libx264', '-preset', 'medium', '-crf', '23',
               '-pix_fmt', 'yuv420p', '-t', '600', str(output)]
    else:
        # Create video with color background
        cmd = ['ffmpeg', '-y', '-f', 'lavfi', '-i',
               'color=c=#0a0a23:s=1920x1080:d=600:rate=30',
               '-vf', "drawtext=text='Relaxing Music':fontsize=48:fontcolor=white:x=(w-text_w)/2:y=(h-text_h)/2",
               '-c:v', 'libx264', '-preset', 'medium', '-crf', '23',
               '-pix_fmt', 'yuv420p', str(output)]
    
    subprocess.run(cmd)
    print(f'Done! File size: {output.stat().st_size / 1024 / 1024:.2f} MB')
    return str(output)

# Run
video = create_video()

In [ ]:
# Download the video
from google.colab import files
files.download(video)